# Hands-On: Polynomial Regression on the Diabetes Dataset

## Objective
Explore how polynomial regression captures non-linear patterns in data. Using the same diabetes dataset, we'll build polynomial models of increasing degree, understand what `PolynomialFeatures` does under the hood, and observe the classic underfitting → good fit → overfitting progression.

## Dataset
- **Source**: sklearn's built-in diabetes dataset
- **Samples**: 442 patients
- **Features**: 10 baseline physiological measurements (age, sex, BMI, blood pressure, 6 blood serum markers)
- **Target**: A quantitative measure of disease progression one year after baseline

## What We'll Cover
1. What `PolynomialFeatures` does — and why it still counts as linear regression
2. How the feature matrix expands when we add polynomial terms
3. Training polynomial models of degree 1 through 5
4. Comparing Train R² vs Test R² across degrees
5. Identifying the degree where overfitting begins

---
## Step 1: Import Libraries

We need one new import compared to the regularisation notebook: `PolynomialFeatures` from `sklearn.preprocessing`.

This transformer takes your original feature matrix and **generates new columns** for every polynomial combination of the input features — so X becomes [X, X², X³, X·Y, ...] depending on the degree you choose.

We also import `Pipeline` to chain the two steps (feature expansion → model fitting) cleanly.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')  # Suppress convergence warnings for high-degree fits

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

---
## Step 2: Load and Explore the Data

Same dataset as before — 442 patients, 10 standardised features, one continuous target.
We print the shape so we know exactly what we're working with before any transformation.

In [2]:
# Load the diabetes dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
feature_names = diabetes.feature_names

print("Dataset Shape:")
print(f"  X (features): {X.shape}  → {X.shape[0]} patients, {X.shape[1]} features")
print(f"  y (target):   {y.shape}")
print(f"\nFeature names: {list(feature_names)}")
print(f"\nTarget range: {y.min():.0f} to {y.max():.0f}")
print(f"Target mean:  {y.mean():.1f}")

Dataset Shape:
  X (features): (442, 10)  → 442 patients, 10 features
  y (target):   (442,)

Feature names: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']

Target range: 25 to 346
Target mean:  152.1


---
## Step 3: Split Data into Training and Test Sets

Identical split to the regularisation notebook — 80% train, 20% test, same random seed.
Keeping this consistent means our results are directly comparable across both notebooks.

> **Important**: The split must happen *before* any PolynomialFeatures transformation.
> If you transform first and split later, information from the test set leaks into training — this is called **data leakage** and will make your model look better than it really is.

In [ ]:
# Split FIRST — always, before any feature transformation
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print(f"\n⚠️  Always split before transforming — never after.")

---
## Step 4: Understanding PolynomialFeatures

Before building any models, let's understand exactly what `PolynomialFeatures` does to our data.

With **degree=2** and 10 original features, it creates:
- All original features: X₁, X₂, ..., X₁₀  
- All squared terms: X₁², X₂², ..., X₁₀²  
- All interaction terms: X₁·X₂, X₁·X₃, ..., X₉·X₁₀  
- A bias column of 1s (the intercept)

The total number of features explodes quickly. The formula is: **C(n+d, d)** where n = original features and d = degree.

We use `include_bias=False` because `LinearRegression` adds its own intercept — we don't need a duplicate column of 1s.

In [ ]:
# Let's see what happens to our feature count at each degree
print("HOW POLYNOMIALFEATURES EXPANDS YOUR DATA")
print("=" * 50)
print(f"{'Degree':<10} {'Original':>10} {'After Transform':>18} {'New Features':>14}")
print("-" * 50)

for degree in [1, 2, 3, 4, 5]:
    # Create transformer but don't fit yet — just check output shape
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X_train)  # Apply to training data
    original = X_train.shape[1]           # Always 10
    after = X_poly.shape[1]               # Grows rapidly
    new = after - original
    print(f"{degree:<10} {original:>10} {after:>18} {new:>14}")

print("\n💡 Key insight: At degree=2, our 10 features become 65.")
print("   The model has many more parameters to fit — overfitting risk rises.")

---
## Step 5: Why It's Still Called 'Linear' Regression

This is one of the most common points of confusion. Polynomial regression *looks* non-linear (it fits curves), but it **is** linear regression under the hood.

The key: **'linear' refers to the relationship between the model and its coefficients (β), not between Y and X.**

Imagine we have one feature X. After PolynomialFeatures(degree=2), our model becomes:

$$Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \varepsilon$$

We rename: let Z₁ = X and Z₂ = X². Now the model is:

$$Y = \beta_0 + \beta_1 Z_1 + \beta_2 Z_2 + \varepsilon$$

That's just multiple linear regression with two features (Z₁ and Z₂). LinearRegression doesn't know or care that Z₂ came from squaring Z₁. It just sees a matrix of numbers and finds the best β.

In [ ]:
# Demonstrate: PolynomialFeatures + LinearRegression = Polynomial Regression
# Let's manually show the two-step process for degree=2

# Step A: Expand features
poly_demo = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly2 = poly_demo.fit_transform(X_train)   # Fit on train, transform train
X_test_poly2  = poly_demo.transform(X_test)        # Only transform test — never fit on test

print("MANUAL TWO-STEP POLYNOMIAL REGRESSION (Degree 2)")
print(f"  Original X_train shape:     {X_train.shape}")
print(f"  After PolynomialFeatures:   {X_train_poly2.shape}")
print(f"  → {X_train_poly2.shape[1]} features now fed to LinearRegression\n")

# Step B: Fit standard LinearRegression on the expanded matrix
lr_manual = LinearRegression()
lr_manual.fit(X_train_poly2, y_train)

train_r2 = lr_manual.score(X_train_poly2, y_train)
test_r2  = lr_manual.score(X_test_poly2,  y_test)

print(f"  Train R²: {train_r2:.4f}")
print(f"  Test R²:  {test_r2:.4f}")
print(f"  Gap:      {train_r2 - test_r2:.4f}")
print(f"\n✓ LinearRegression fitted {X_train_poly2.shape[1]} coefficients — one per expanded feature")
print(f"  It has no idea X²₁ came from squaring X₁. It just minimises MSE.")

---
## Step 6: Build a Pipeline for Clean Implementation

Doing the two-step manually works, but a **Pipeline** is cleaner and safer:

1. It guarantees `PolynomialFeatures` is fitted *only* on training data
2. It applies the transformation automatically when you call `.predict()` or `.score()`
3. It prevents data leakage — the single most common mistake in feature preprocessing

We also add a `StandardScaler` step. Polynomial features can have vastly different scales —  
X² might be tiny while X⁵ is enormous. Scaling prevents numerical instability.

In [ ]:
def build_poly_pipeline(degree):
    """
    Build a Pipeline with three steps:
      1. PolynomialFeatures  → expand X into X, X², X³... up to 'degree'
      2. StandardScaler      → rescale all expanded features to mean=0, std=1
      3. LinearRegression    → fit β coefficients on the scaled, expanded matrix
    
    include_bias=False: LinearRegression adds its own intercept, no duplicate needed
    """
    return Pipeline([
        ('poly',   PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler', StandardScaler()),          # Rescale after expansion
        ('model',  LinearRegression())         # Standard least-squares regression
    ])

print("Pipeline structure for degree=2:")
print(build_poly_pipeline(2))
print("\n💡 Calling pipeline.fit(X_train, y_train) runs all three steps in order.")
print("   Calling pipeline.score(X_test, y_test) applies steps 1 & 2 to X_test first,")
print("   using the parameters learned from training — no re-fitting on test data.")

---
## Step 7: Train Polynomial Models — Degree 1 Through 5

Now we train one pipeline per degree and record Train R² and Test R² for each.

What to watch for:
- **Train R² should increase** as degree increases — more flexibility = better fit on training data
- **Test R² should peak** at some degree and then drop — that's the overfitting signal
- **The gap (Train - Test)** should widen as overfitting sets in

In [ ]:
degrees = [1, 2, 3, 4, 5]
results = {}   # Store {degree: {'train_r2': ..., 'test_r2': ..., 'pipeline': ...}}

print("POLYNOMIAL REGRESSION — DEGREE COMPARISON")
print("=" * 60)
print(f"{'Degree':<10} {'Train R²':>10} {'Test R²':>10} {'Gap':>10} {'Features':>10}")
print("-" * 60)

for d in degrees:
    pipe = build_poly_pipeline(d)    # Build fresh pipeline for this degree
    pipe.fit(X_train, y_train)       # Fit: expand → scale → regress
    
    train_r2 = pipe.score(X_train, y_train)
    test_r2  = pipe.score(X_test,  y_test)
    gap      = train_r2 - test_r2
    n_feats  = pipe.named_steps['poly'].n_output_features_
    
    results[d] = {
        'train_r2': train_r2,
        'test_r2':  test_r2,
        'pipeline': pipe
    }
    
    # Flag overfitting visually
    flag = " ← overfitting" if gap > 0.15 else ""
    print(f"{d:<10} {train_r2:>10.4f} {test_r2:>10.4f} {gap:>10.4f} {n_feats:>10}{flag}")

# Identify best degree based on test R²
best_degree = max(results, key=lambda d: results[d]['test_r2'])
print(f"\n✓ Best degree by Test R²: {best_degree}  "
      f"(Test R² = {results[best_degree]['test_r2']:.4f})")

---
## Step 8: Visualise the Bias-Variance Tradeoff

The table above is informative, but a plot makes the pattern unmistakable.

The **model complexity curve** shows:
- Training error (blue line) — always improves as degree increases
- Test error (orange line) — improves up to a point, then gets worse

The degree where the two lines diverge most sharply is where overfitting begins. The degree just before that divergence is usually your sweet spot.

In [ ]:
train_scores = [results[d]['train_r2'] for d in degrees]
test_scores  = [results[d]['test_r2']  for d in degrees]

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(degrees, train_scores, 'o-', color='steelblue',  linewidth=2,
        markersize=8, label='Train R²')
ax.plot(degrees, test_scores,  's-', color='darkorange', linewidth=2,
        markersize=8, label='Test R²')

# Mark the best test degree
ax.axvline(x=best_degree, color='green', linestyle='--', alpha=0.6,
           label=f'Best degree = {best_degree}')

ax.set_xlabel('Polynomial Degree', fontsize=12)
ax.set_ylabel('R² Score', fontsize=12)
ax.set_title('Bias-Variance Tradeoff: Training vs Test R²\n'
             'Diabetes Dataset — Polynomial Regression', fontsize=13)
ax.set_xticks(degrees)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Annotate the divergence region
ax.annotate('Overfitting region →\ngap widens here',
            xy=(best_degree + 0.3, test_scores[best_degree - 1] - 0.05),
            fontsize=9, color='red', alpha=0.8)

plt.tight_layout()
plt.show()

print("\n💡 Notice: Train R² keeps climbing. Test R² peaks and then falls.")
print("   That divergence IS the bias-variance tradeoff in action.")

---
## Step 9: Inspect the Best-Degree Model's Coefficients

After the pipeline expands and scales our features, `LinearRegression` fits one coefficient per expanded feature.

We can extract these and see which polynomial terms the model weighted most heavily. Large positive coefficients push the prediction up; large negative ones pull it down.

We'll look at the **top 15 most influential terms** from the best-degree model.

In [ ]:
best_pipe = results[best_degree]['pipeline']

# Get feature names that PolynomialFeatures created
poly_step    = best_pipe.named_steps['poly']
model_step   = best_pipe.named_steps['model']
poly_feature_names = poly_step.get_feature_names_out(feature_names)

# Pair each feature name with its coefficient
coef_pairs = list(zip(poly_feature_names, model_step.coef_))

# Sort by absolute value — largest influence at the top
coef_pairs_sorted = sorted(coef_pairs, key=lambda x: abs(x[1]), reverse=True)

print(f"TOP 15 MOST INFLUENTIAL TERMS — Degree {best_degree} Model")
print("=" * 55)
print(f"{'Feature':<25} {'Coefficient':>12} {'Direction':>12}")
print("-" * 55)

for feat, coef in coef_pairs_sorted[:15]:
    direction = "↑ progression" if coef > 0 else "↓ progression"
    print(f"{feat:<25} {coef:>12.2f} {direction:>12}")

print(f"\n  ... and {len(coef_pairs) - 15} more terms")
print(f"\n✓ Total polynomial features in degree-{best_degree} model: {len(coef_pairs)}")

---
## Step 10: Actual vs Predicted — Best Degree Model

A quick sanity check: plot actual target values against what our best-degree model predicted.

What a good model looks like on this chart:
- Points clustered tightly around the **45-degree diagonal line** (predicted ≈ actual)
- No strong curve or fan shape in the residuals
- Roughly equal scatter above and below the line

In [ ]:
y_pred = best_pipe.predict(X_test)    # Generate predictions on held-out test data

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Plot 1: Actual vs Predicted ──────────────────────────────────────────────
ax1 = axes[0]
ax1.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white',
            linewidth=0.5, s=60)

# Perfect prediction line
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax1.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')

ax1.set_xlabel('Actual Progression Score', fontsize=11)
ax1.set_ylabel('Predicted Progression Score', fontsize=11)
ax1.set_title(f'Actual vs Predicted\nDegree {best_degree} — Test R² = '
              f'{results[best_degree]["test_r2"]:.3f}', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Residuals ─────────────────────────────────────────────────────────
residuals = y_test - y_pred   # Actual minus predicted

ax2 = axes[1]
ax2.scatter(y_pred, residuals, alpha=0.6, color='darkorange', edgecolors='white',
            linewidth=0.5, s=60)
ax2.axhline(y=0, color='red', linestyle='--', linewidth=1.5, label='Zero error line')

ax2.set_xlabel('Predicted Progression Score', fontsize=11)
ax2.set_ylabel('Residual (Actual − Predicted)', fontsize=11)
ax2.set_title(f'Residual Plot\nRandom scatter = well-behaved model', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"\nTest RMSE: {rmse:.2f}  (average prediction is off by ~{rmse:.0f} progression points)")
print("💡 Residuals should look like random noise with no clear pattern.")
print("   A curved residual pattern would signal a still-missing non-linearity.")

---
## Step 11: Effect of Degree on MSE — Full Picture

Let's look at MSE alongside R² to confirm the same story from a different angle.
MSE is in the original units of the target — so it's more intuitive for communicating with non-technical stakeholders.

In [ ]:
print("FULL METRICS TABLE — ALL DEGREES")
print("=" * 75)
print(f"{'Degree':<8} {'Train R²':>10} {'Test R²':>10} {'Train MSE':>12} "
      f"{'Test MSE':>12} {'Gap R²':>10}")
print("-" * 75)

for d in degrees:
    pipe      = results[d]['pipeline']
    tr2       = results[d]['train_r2']
    te2       = results[d]['test_r2']
    tr_mse    = mean_squared_error(y_train, pipe.predict(X_train))
    te_mse    = mean_squared_error(y_test,  pipe.predict(X_test))
    gap       = tr2 - te2
    marker    = " ✓ best" if d == best_degree else ""
    print(f"{d:<8} {tr2:>10.4f} {te2:>10.4f} {tr_mse:>12.1f} "
          f"{te_mse:>12.1f} {gap:>10.4f}{marker}")

print("\n💡 Train MSE always falls. Test MSE eventually rises — that's overfitting.")
print(f"   Degree {best_degree} gives the lowest Test MSE and best Test R².")

---
## Summary & Key Findings

### What We Did:
1. Used `PolynomialFeatures` to expand 10 features into polynomial and interaction terms
2. Wrapped the expansion in a `Pipeline` alongside `StandardScaler` and `LinearRegression`
3. Trained models from degree 1 through 5 and measured Train R² and Test R² at each step

### What We Learned:

1. **PolynomialFeatures doesn't change the algorithm — only the inputs**  
   LinearRegression still minimises MSE using the same least-squares formula. It just receives a richer, expanded feature matrix.

2. **Train R² always improves with degree**  
   More parameters → better fit on seen data. This is expected and not a reason to celebrate.

3. **Test R² peaks and then falls**  
   That peak is your optimal degree. Beyond it, the model starts memorising training noise.

4. **Feature count explodes fast**  
   10 features at degree 2 → 65 features. At degree 3 → 286. This is why scaling and regularisation (Ridge/Lasso) become critical companions to polynomial regression.

5. **Always use a Pipeline**  
   It prevents data leakage and makes the preprocessing reproducible and deployment-safe.

### Next Step:
> Combine both notebooks: apply **Ridge or Lasso regularisation on top of polynomial features** to get the flexibility of curves with the overfitting protection of regularisation — the best of both worlds.

In [ ]:
print("=" * 50)
print("EXERCISE COMPLETE!")
print("=" * 50)
print("\nKey Questions Answered:")
print("  ✓ What does PolynomialFeatures actually do to the data?")
print("  ✓ Why is polynomial regression still called 'linear'?")
print("  ✓ How many features do we get at each degree?")
print("  ✓ At what degree does overfitting kick in?")
print("  ✓ Which degree gives the best test performance?")
print(f"\n  Best degree on this dataset: {best_degree}")
print(f"  Test R² at best degree:      {results[best_degree]['test_r2']:.4f}")